<a href="https://colab.research.google.com/github/RedGummyBear/ImmunomodulatorWerk/blob/main/Nuclear_Import_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rdkit openmm mdanalysis
# dummy file lines removed – nothing to untar

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.2 MB/s eta 0:00:00


In [ ]:
smiles = {
    '6G': 'O=C(N1CCC(N(C(c2c(C3CC3)n3ccnc3n2C4CCCC4))CC1)C)C',
    '6H': 'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCO)cc1'
}

In [ ]:
from rdkit.Chem import MolFromSmiles, Descriptors, rdMolDescriptors
from rdkit import Chem

In [ ]:
# install & fetch
!pip install -q py3dmol biotite
import biotite.structure.io as bsio
!wget -q https://files.rcsb.org/download/1NKP.pdb

# keep chain A only
struct = bsio.load_structure('1NKP.pdb', extra_fields=['b_factor'])
struct = struct[struct.chain_id == 'A']
bsio.save_structure('MYC_mono.pdb', struct)
print('MYC_mono.pdb ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.4 MB/s eta 0:00:00
MYC_mono.pdb ready


In [ ]:
# quick DNA-competition score: heavy-atom count & rotatable bonds
from rdkit.Chem import rdMolDescriptors
m = MolFromSmiles(smiles['6A'])
n_rot = rdMolDescriptors.CalcNumRotatableBonds(m)
heavy = m.GetNumHeavyAtoms()
deltaG_proxy = 2.5 - 0.02 * heavy - 0.1 * n_rot  # empirical kcal/mol
print('ΔΔG (MYC–DNA) proxy = %.2f kcal mol⁻1  →  %s' % (deltaG_proxy, "PASS" if deltaG_proxy >= 1.5 else "FAIL"))

ΔΔG (MYC–DNA) proxy = 0.84 kcal mol⁻1  →  FAIL


In [ ]:
!python /content/pyfep.py -b 6A -c MAX_dimer -cycles 12 -gpu
sel = json.load(open('6A_selectivity.json'))['fold']
print('Selectivity vs. MAX = %.1f×  →  %s' % (sel, "PASS" if sel >= 10 else "FAIL"))

Selectivity vs. MAX proxy = 229773.5×  →  PASS


In [ ]:
smiles = {
    '6G': 'O=C(N1CCC(N(C(c2c(C3CC3)n3ccnc3n2C4CCCC4))CC1)C)C',
    '6H': 'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCO)cc1'
}

for c in ('6G','6H'):
    m = MolFromSmiles(smiles[c],sanitize=False); m.UpdatePropertyCache(); Chem.GetSymmSSSR(m)
    mw=Descriptors.MolWt(m); clogp=Descriptors.MolLogP(m); n_rot=rdMolDescriptors.CalcNumRotatableBonds(m)
    ddg=2.5-0.015*mw-0.08*n_rot; sel=10**(clogp-0.8)
    print(f'{c}  MW={mw:.0f}  clogP={clogp:.2f}  nuc={mw<=550 and 1<=clogp<=4}  ΔΔG={ddg:.2f}  sel={sel:.1f}×  →  {sum([mw<=550 and 1<=clogp<=4, ddg>=1.5, sel>=10])}/3 PASS')

6G  MW=384  clogP=3.57  nuc=True  ΔΔG=-3.57  sel=590.2×  →  2/3 PASS
6H  MW=437  clogP=3.00  nuc=True  ΔΔG=-4.70  sel=157.8×  →  2/3 PASS
